# ParallelFindAllTool

`ParallelFindAllTool` exposes the Parallel [FindAll API](https://docs.parallel.ai/findall-api/overview) — entity discovery against a natural-language objective and a set of boolean match conditions. Returns ranked candidates that satisfy all conditions.

Generators: `preview` (free, capped at 10 results), then `base` / `core` / `pro` for higher-quality runs.

## Setup

In [ ]:
%pip install --quiet -U langchain-parallel

In [ ]:
import getpass
import os

if not os.environ.get("PARALLEL_API_KEY"):
    os.environ["PARALLEL_API_KEY"] = getpass.getpass("Parallel API key:\n")


## Quick discovery (preview generator)

The `preview` generator is the fastest way to sanity-check your conditions. It's capped at 10 candidates.

In [ ]:
from langchain_parallel import (
    FindAllMatchCondition,
    ParallelFindAllTool,
)

tool = ParallelFindAllTool()

result = await tool.ainvoke({
    "objective": "Pure-play public LLM API providers",
    "entity_type": "company",
    "match_conditions": [
        FindAllMatchCondition(
            name="public_us",
            description="Company is publicly traded on a US exchange",
        ).model_dump(),
        FindAllMatchCondition(
            name="llm_api_revenue",
            description="Primary revenue is selling LLM inference via API",
        ).model_dump(),
    ],
    "generator": "preview",
    "match_limit": 5,
})

for c in result.get("candidates", []):
    print(c.get("name"), "-", c.get("url"))


## Higher-quality runs

For non-preview runs, use `match_limit` 5..1000 with `generator="base"` / `"core"` / `"pro"`. These take longer (minutes); the tool polls under the hood and returns when status is terminal (`completed` / `cancelled` / `failed`).

```python
result = await tool.ainvoke({
    "objective": "Independent solar-installer companies based in the EU",
    "match_conditions": [
        FindAllMatchCondition(description="Headquartered in an EU country").model_dump(),
        FindAllMatchCondition(description="Primarily installs residential solar PV").model_dump(),
    ],
    "generator": "core",
    "match_limit": 50,
})
```

You can also exclude entries you've already seen via `FindAllExcludeEntry(url=...)`.

## Cancel a long-running search

In [ ]:
# Available cancellation API (no live call here so the notebook stays fast):
print("public methods:", [m for m in dir(tool) if not m.startswith("_") and "cancel" in m])


## API reference

- [FindAll API guides](https://docs.parallel.ai/findall-api/overview)